# Week 1 Task - Titanic Data Cleaning & EDA

**Goal:** Acquire a public dataset, clean it, and explore it with basic stats and visualizations.

**Dataset:** Titanic passenger dataset (891 rows, 12 columns), from the public GitHub mirror:
https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv


## 1. Load the data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('../data/titanic_raw.csv')
print(df.shape)
df.head()


## 2. Check for missing values, duplicates, and data types

In [ ]:
print(df.isnull().sum())
print()
print('Duplicate rows:', df.duplicated().sum())
print()
print(df.dtypes)


Findings:
- `Cabin` is missing about 77% of its values.
- `Age` is missing about 20% of its values.
- `Embarked` is missing only 2 values.
- No duplicate rows.


## 3. Clean the data

In [ ]:
# drop Cabin - too many missing values to be useful
df_clean = df.drop(columns=['Cabin'])

# fill missing Age with the median (robust to outliers)
median_age = df_clean['Age'].median()
df_clean['Age'] = df_clean['Age'].fillna(median_age)

# fill missing Embarked with the mode (only 2 rows affected)
mode_embarked = df_clean['Embarked'].mode()[0]
df_clean['Embarked'] = df_clean['Embarked'].fillna(mode_embarked)

# drop any duplicate rows just as a safety step
df_clean = df_clean.drop_duplicates()

# fix data types - these are categories, not raw numbers
df_clean['Survived'] = df_clean['Survived'].astype('category')
df_clean['Pclass']   = df_clean['Pclass'].astype('category')
df_clean['Sex']      = df_clean['Sex'].astype('category')
df_clean['Embarked'] = df_clean['Embarked'].astype('category')

# engineer a FamilySize feature
df_clean['FamilySize'] = df_clean['SibSp'] + df_clean['Parch'] + 1

print(df_clean.isnull().sum())
df_clean.to_csv('../data/titanic_clean.csv', index=False)


## 4. Summary statistics

In [ ]:
print(df_clean.describe())
print()
print(df_clean['Survived'].value_counts(normalize=True) * 100)


## 5. Visualizations

### 5.1 Missing values before cleaning

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

plt.figure(figsize=(6,4))
plt.bar(missing.index, missing.values, color='indianred')
plt.title('Missing Values Before Cleaning')
plt.ylabel('Number of missing rows')
plt.xlabel('Column')
plt.tight_layout()
plt.savefig('../images/1_missing_values.png', dpi=150)
plt.show()


### 5.2 Age distribution

In [ ]:
plt.figure(figsize=(6,4))
plt.hist(df_clean['Age'], bins=25, color='steelblue', edgecolor='black')
plt.title('Distribution of Passenger Age')
plt.xlabel('Age')
plt.ylabel('Number of passengers')
plt.tight_layout()
plt.savefig('../images/2_age_histogram.png', dpi=150)
plt.show()


### 5.3 Survival rate by passenger class

In [ ]:
surv_by_class = df_clean.groupby('Pclass')['Survived'].mean() * 100

plt.figure(figsize=(6,4))
plt.bar(surv_by_class.index.astype(str), surv_by_class.values, color=['#4c72b0','#dd8452','#55a868'])
plt.title('Survival Rate by Passenger Class')
plt.xlabel('Passenger Class')
plt.ylabel('Survival Rate (%)')
plt.tight_layout()
plt.savefig('../images/3_survival_by_class.png', dpi=150)
plt.show()


### 5.4 Fare paid: survivors vs non-survivors

In [ ]:
data_to_plot = [df_clean[df_clean['Survived']==0]['Fare'], df_clean[df_clean['Survived']==1]['Fare']]

plt.figure(figsize=(6,4))
plt.boxplot(data_to_plot, tick_labels=['Died', 'Survived'])
plt.title('Fare Paid: Survivors vs Non-Survivors')
plt.ylabel('Fare')
plt.tight_layout()
plt.savefig('../images/4_fare_boxplot.png', dpi=150)
plt.show()


### 5.5 Correlation heatmap

In [ ]:
num_df = df_clean[['Survived','Pclass','Age','SibSp','Parch','Fare','FamilySize']].copy()
num_df['Survived'] = num_df['Survived'].astype(int)
num_df['Pclass'] = num_df['Pclass'].astype(int)
corr = num_df.corr()

plt.figure(figsize=(6,5))
im = plt.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im)
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha='right')
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title('Correlation Heatmap of Numeric Features')
plt.tight_layout()
plt.savefig('../images/5_correlation_heatmap.png', dpi=150)
plt.show()


## 6. Insights

- **Sex** was the biggest factor in survival: ~74% of female passengers survived vs ~19% of male passengers.
- **Passenger class** mattered a lot: 63% (1st class) vs 47% (2nd class) vs 24% (3rd class) survival rate.
- **Fare and Pclass** are strongly (negatively) correlated, and Fare has a modest positive correlation with survival.
- **Family size** had a non-linear relationship with survival - small families (2-4 people) survived at higher rates than solo travelers or very large families.
- The **Cabin** column was too incomplete (77% missing) to be usable, so it was dropped.
